# UFO Review Workbench

This notebook is the main analysis entry point for the local `war.gov` UFO corpus.

It is organized to do four things:
1. Load the important local artifacts.
2. Inspect records, documents, and page-based chunks.
3. Surface OCR/native extraction quality issues.
4. Provide a skeleton for the next LLM review stage.


In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

ROOT = Path.cwd()
DATASET_ROOT = ROOT / 'data' / 'war_gov_ufo_release_01'
CORPUS_ROOT = DATASET_ROOT / 'corpus'
EXTRACTED_ROOT = DATASET_ROOT / 'extracted'
OCR_ROOT = EXTRACTED_ROOT / 'ocr'

PATHS = {
    'download_summary': DATASET_ROOT / 'summary.json',
    'records': DATASET_ROOT / 'records.json',
    'downloads': DATASET_ROOT / 'downloads.json',
    'native_summary': EXTRACTED_ROOT / 'summary.json',
    'native_manifest': EXTRACTED_ROOT / 'manifest.json',
    'ocr_queue': EXTRACTED_ROOT / 'ocr_queue.json',
    'corpus_summary': CORPUS_ROOT / 'summary.json',
    'documents': CORPUS_ROOT / 'documents.json',
    'chunks_jsonl': CORPUS_ROOT / 'chunks.jsonl',
}

for name, path in PATHS.items():
    print(f'{name:16} -> {path}')


In [ ]:
def load_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def load_jsonl(path: Path):
    rows = []
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

download_summary = load_json(PATHS['download_summary'])
records = load_json(PATHS['records'])
downloads = load_json(PATHS['downloads'])
native_summary = load_json(PATHS['native_summary'])
native_manifest = load_json(PATHS['native_manifest'])
ocr_queue = load_json(PATHS['ocr_queue'])
corpus_summary = load_json(PATHS['corpus_summary'])
documents = load_json(PATHS['documents'])
chunks = load_jsonl(PATHS['chunks_jsonl'])

print('Loaded:')
print('  records   ', len(records))
print('  downloads ', len(downloads))
print('  documents ', len(documents))
print('  chunks    ', len(chunks))


## Snapshot

These are the top-level health metrics for the current local corpus.

In [ ]:
snapshot = {
    'download_summary': download_summary,
    'native_summary': native_summary,
    'corpus_summary': corpus_summary,
}
snapshot


In [ ]:
if pd is not None:
    pd.DataFrame(documents).head(10)
else:
    documents[:3]


## Document-Level Inspection

One row here is one source asset. This is the right place to inspect extraction method, page counts, and missing-text cases.

In [ ]:
doc_method_counts = Counter(doc['extraction_method'] for doc in documents)
doc_kind_counts = Counter(doc['asset_kind'] for doc in documents)
doc_method_counts, doc_kind_counts


In [ ]:
docs_without_text = [doc for doc in documents if not doc['has_text']]
len(docs_without_text), docs_without_text[:5]


In [ ]:
if pd is not None:
    (
        pd.DataFrame(documents)
        .sort_values(['text_char_count', 'page_count'], ascending=[False, False])
        [['asset_id', 'title', 'agency', 'extraction_method', 'page_count', 'text_char_count', 'source_path']]
        .head(20)
    )
else:
    sorted(documents, key=lambda d: (d['text_char_count'], d.get('page_count') or 0), reverse=True)[:5]


## Page-Based Chunks

Each chunk now represents exactly one page when page-level text exists.

In [ ]:
chunks[0]


In [ ]:
chunk_counts_by_doc = Counter(chunk['asset_id'] for chunk in chunks)
sorted(chunk_counts_by_doc.items(), key=lambda x: x[1], reverse=True)[:15]


In [ ]:
if pd is not None:
    (
        pd.DataFrame(chunks)
        [['asset_id', 'page_number', 'char_count', 'word_count', 'extraction_method', 'title']]
        .head(20)
    )
else:
    chunks[:5]


## Quick Lookup Helpers

These helpers make it easy to pivot between records, documents, and page chunks.

In [ ]:
records_by_index = {record['record_index']: record for record in records}
documents_by_asset = {doc['asset_id']: doc for doc in documents}

def get_document(asset_id: str):
    return documents_by_asset[asset_id]

def get_chunks_for_asset(asset_id: str):
    return [chunk for chunk in chunks if chunk['asset_id'] == asset_id]

def preview_asset(asset_id: str, pages: int = 3):
    doc = get_document(asset_id)
    selected = get_chunks_for_asset(asset_id)[:pages]
    print('ASSET:', asset_id)
    print('TITLE:', doc['title'])
    print('METHOD:', doc['extraction_method'])
    print('SOURCE:', doc['source_path'])
    print()
    for chunk in selected:
        print(f"--- Page {chunk['page_number']} ---")
        print(chunk['text'][:1200])
        print()


In [ ]:
preview_asset('143_nasa-uap-d6_apollo_17_technical_crew_debriefing_1973', pages=2)


## OCR Quality Triage

A simple first-pass way to identify pages that may be low-signal or messy.

In [ ]:
low_signal_chunks = [
    chunk for chunk in chunks
    if chunk['extraction_method'] == 'ocr' and chunk['char_count'] < 120
]
len(low_signal_chunks), low_signal_chunks[:10]


In [ ]:
if pd is not None:
    pd.DataFrame(low_signal_chunks)[['asset_id', 'page_number', 'char_count', 'title']].head(25)
else:
    low_signal_chunks[:5]


## LLM Review Skeleton

This section is a scaffold for the next model-review stage.

Important: the exact model ID for the OpenAI OSS 120B model should be filled in explicitly once you decide the final serving/runtime path. I left a placeholder instead of guessing the exact ID.


In [ ]:
# Fill this in with the exact model identifier you plan to use.
MODEL_NAME = 'TODO_SET_EXACT_OSS_120B_MODEL_ID'

# Optional runtime config.
TEMPERATURE = 0.2
MAX_OUTPUT_TOKENS = 1200

SYSTEM_PROMPT = '''You are reviewing declassified UFO/UAP-related source material.
Work conservatively.
Separate direct evidence from speculation.
Call out dates, places, agencies, witnesses, and anomalous claims.
If OCR looks unreliable, say so explicitly.
Return structured JSON only.
'''

USER_TEMPLATE = '''Review the following page chunk from the UFO corpus.

Metadata:
- asset_id: {asset_id}
- title: {title}
- agency: {agency}
- release_date: {release_date}
- incident_date: {incident_date}
- incident_location: {incident_location}
- page_number: {page_number}
- extraction_method: {extraction_method}

Text:
{text}

Return JSON with:
- summary
- entities
- claims
- anomaly_signals
- confidence
- ocr_quality_notes
- follow_up_pages
'''


In [ ]:
def build_review_payload(chunk: dict) -> dict:
    return {
        'model': MODEL_NAME,
        'temperature': TEMPERATURE,
        'max_output_tokens': MAX_OUTPUT_TOKENS,
        'input': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': USER_TEMPLATE.format(
                    asset_id=chunk['asset_id'],
                    title=chunk['title'],
                    agency=chunk['agency'],
                    release_date=chunk['release_date'],
                    incident_date=chunk['incident_date'],
                    incident_location=chunk['incident_location'],
                    page_number=chunk['page_number'],
                    extraction_method=chunk['extraction_method'],
                    text=chunk['text'],
                ),
            },
        ],
    }

example_payload = build_review_payload(chunks[0])
example_payload


In [ ]:
def select_review_candidates(
    chunks: list[dict],
    min_chars: int = 300,
    methods: tuple[str, ...] = ('ocr', 'native'),
    limit: int = 25,
):
    selected = [
        chunk for chunk in chunks
        if chunk['char_count'] >= min_chars and chunk['extraction_method'] in methods
    ]
    return selected[:limit]

review_candidates = select_review_candidates(chunks, min_chars=400, methods=('ocr', 'native'), limit=10)
len(review_candidates), review_candidates[:2]


## Batch Review Skeleton

This is intentionally left as a dry-run skeleton. Wire it to your chosen client/runtime once you lock in the exact model endpoint.

In [ ]:
def dry_run_review_batch(candidates: list[dict]):
    jobs = []
    for chunk in candidates:
        jobs.append({
            'chunk_id': chunk['chunk_id'],
            'asset_id': chunk['asset_id'],
            'page_number': chunk['page_number'],
            'payload': build_review_payload(chunk),
        })
    return jobs

jobs = dry_run_review_batch(review_candidates)
jobs[0]


In [ ]:
# Example shape for storing model outputs once connected.
REVIEW_RESULT_TEMPLATE = {
    'chunk_id': 'asset::page::0001',
    'model': MODEL_NAME,
    'summary': '',
    'entities': [],
    'claims': [],
    'anomaly_signals': [],
    'confidence': None,
    'ocr_quality_notes': '',
    'follow_up_pages': [],
    'raw_response': None,
}
REVIEW_RESULT_TEMPLATE
